# Physics for Scientists & Engineers

<img src="..//pics/cover.jpg" width=400> 

In [1]:
import vpython as vp
from astropy import units as u
from astropy import constants as const
import sympy as sp
import numpy as np
from IPython.display import display, Math


<IPython.core.display.Javascript object>

### Problems
#### SECTION Challenge Problems
#### Problem 49
page 105

<img src="..//pics/FigP4_49.png" width=400> 

A skier leaves the ramp of a ski jump with a velocity
of v = 10.0 m/s at $ \theta = 15.0°$ above the horizontal as
shown in Figure P4.49. The slope where she will land
is inclined downward at $\phi = 50.0°$, and air resistance
is negligible. 

Find  (a) the distance from the end of the
ramp to where the jumper lands and (b) her velocity
components just before the landing. 

<img src="..//pics/FigP4_49_ans.png" width=400> 


In [ ]:
# Define symbolic variables
v0, theta, phi, g, t, d = sp.symbols("v0 theta phi g t d", real=True, positive=True)

# 1. Trajectory equations
x_t = v0 * sp.cos(theta) * t
y_t = v0 * sp.sin(theta) * t - sp.Rational(1, 2) * g * t**2

# 2. Time of flight to reach the slope
t_land = (d * sp.cos(phi)) / (v0 * sp.cos(theta))

# 3. Solve for landing distance (d) along the incline
y_eq = sp.Eq(-d * sp.sin(phi), y_t.subs(t, t_land))
d_expr = sp.simplify(sp.solve(y_eq, d)[0])

# 4. Landing velocity components
vx_expr = v0 * sp.cos(theta)
vy_expr = sp.simplify(v0 * sp.sin(theta) - g * t_land.subs(d, d_expr))

print("--- Symbolic Formula for Distance (d) ---")
display(Math(rf"d = {sp.latex(d_expr)}"))

print("\n--- Symbolic Formula for Vertical Velocity (v_y) ---")
display(Math(rf"v_y = {sp.latex(vy_expr)}"))

--- Symbolic Formula for Distance (d) ---


<IPython.core.display.Math object>


--- Symbolic Formula for Vertical Velocity (v_y) ---


<IPython.core.display.Math object>

In [ ]:
# =====================================================================
# 1. LAMBDIFY THE SYMPY EXPRESSIONS
# =====================================================================
# Convert the symbolic expressions (d_expr, vx_expr, vy_expr) into NumPy-backed functions
f_d = sp.lambdify((v0, theta, phi, g), d_expr, modules="numpy")
f_vx = sp.lambdify((v0, theta), vx_expr, modules="numpy")
f_vy = sp.lambdify((v0, theta, phi, g), vy_expr, modules="numpy")


# =====================================================================
# 2. DEFINE INPUTS WITH ASTROPY UNITS
# =====================================================================
v0_val = 10.0 * (u.m / u.s)
theta_val = 15.0 * u.deg
phi_val = 50.0 * u.deg
g_val = 9.80 * (u.m / u.s**2)

# Convert angle units to radians for numpy evaluation
theta_rad = theta_val.to(u.rad).value
phi_rad = phi_val.to(u.rad).value


# =====================================================================
# 3. EVALUATE LAMBDIFIED FUNCTIONS & RE-ATTACH UNITS
# =====================================================================
# Part (a): Distance along slope
d_res = f_d(v0_val.value, theta_rad, phi_rad, g_val.value) * u.m

# Part (b): Velocity components just before landing
vx_res = f_vx(v0_val.value, theta_rad) * (u.m / u.s)
vy_res = f_vy(v0_val.value, theta_rad, phi_rad, g_val.value) * (u.m / u.s)
v_total_res = np.sqrt(vx_res**2 + vy_res**2)

# Flight time derived from distance
t_flight = (d_res * np.cos(phi_val)) / (v0_val * np.cos(theta_val))

print("--- Part (a) Distance Results (AstroPy) ---")
print(f"Time of flight (t)       : {t_flight.to(u.s):.3f}")
print(f"Landing distance (d)     : {d_res:.2f}")

print("\n--- Part (b) Velocity Component Results (AstroPy) ---")
print(f"Horizontal velocity (v_x): {vx_res:.2f}")
print(f"Vertical velocity (v_y)  : {vy_res:.2f}")
print(f"Total impact speed       : {v_total_res:.2f}")

--- Part (a) Distance Results (AstroPy) ---
Time of flight (t)       : 2.877 s
Landing distance (d)     : 43.24 m

--- Part (b) Velocity Component Results (AstroPy) ---
Horizontal velocity (v_x): 9.66 m / s
Vertical velocity (v_y)  : -25.61 m / s
Total impact speed       : 27.37 m / s
